# 00 · Setup and welcome

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/00-setup-and-data.ipynb)

*setup · 5 min*

> 🇪🇸 **Preparación y bienvenida** — Carga todos los conjuntos de datos y confirma que tu entorno funciona antes de empezar.

Load every dataset and confirm your runtime works before anything else.

## What you will be able to do

- Confirm your Colab runtime can reach every dataset the workshop uses.
- Know which data ships inside the libraries and which is downloaded.
- Recognise the shapes you will be working with all day.

> 🇪🇸 **Lo que podrás hacer:**
>
> - Confirmar que tu entorno de Colab puede acceder a todos los conjuntos de datos que usa el taller.
> - Saber qué datos vienen incluidos en las librerías y cuáles se descargan.
> - Reconocer las formas de los tensores con las que trabajarás durante todo el taller.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
# Section 05 decodes a real video and Colab does not reliably ship an
# ffmpeg backend, so install it now. Everything else below is already here.
%pip install -q "imageio[ffmpeg]"

import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_breast_cancer
from skimage import data
from scipy import signal
from scipy.linalg import lu, toeplitz

HOUSING = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
TAXIS   = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
FLIGHTS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv"

housing = pd.read_csv(HOUSING)
taxis   = pd.read_csv(TAXIS)
flights = pd.read_csv(FLIGHTS)
print(housing.shape, taxis.shape, flights.shape)   # (20640, 10) (6433, 14) (144, 3)

# Section 05's video is 5.9 MB, so don't pull it now -- just prove the backend
# imports and the host answers. Better to find out here than in two hours.
import imageio_ffmpeg, urllib.request
VIDEO_URL = ("https://upload.wikimedia.org/wikipedia/commons/1/1e/"
             "Tormenta_en_l%27Almadrava.webm")
req = urllib.request.Request(VIDEO_URL, headers={
    "User-Agent": "tensors-workshop/1.0 "
                  "(https://github.com/project-delphi/tensors-workshop)",
    "Range": "bytes=0-1023"})
got = urllib.request.urlopen(req, timeout=30).read()
# Assert rather than print the length: a proxy that ignores Range would quietly
# pull all 5.9 MB here and still look like a pass, which is the opposite of what
# this cell is for.
assert len(got) == 1024, f"expected a 1 KB range, got {len(got)} bytes"
print("1024 bytes of video reachable | ffmpeg", imageio_ffmpeg.get_ffmpeg_version())

## Four words before we start / Cuatro palabras antes de empezar

You do **not** need a technical background to use this notebook. Keep these four ideas in mind:

- **Runtime:** the temporary computer that Google Colab gives you to run Python.
- **Dataset:** a collection of examples or measurements, similar to a table in Excel.
- **Shape:** a short description of how the data is arranged. For a table, `(20640, 10)` means **20,640 rows and 10 columns**.
- **Axis:** one direction in the data. In a table, one axis counts rows and another counts columns. In an image, axes can count height, width, and colour.

Think of `shape` as the **label on a box**: it tells you how the contents are arranged, but not what those contents mean.

> 🇪🇸 No necesitas una formación técnica para usar este cuaderno. Conserva estas cuatro ideas:
>
> - **Entorno de ejecución (runtime):** el computador temporal que Google Colab te presta para ejecutar Python.
> - **Conjunto de datos (dataset):** una colección de ejemplos o mediciones, parecida a una tabla de Excel.
> - **Forma (shape):** una descripción corta de cómo están organizados los datos. En una tabla, `(20640, 10)` significa **20.640 filas y 10 columnas**.
> - **Eje (axis):** una dirección de los datos. En una tabla, un eje cuenta filas y otro columnas. En una imagen, los ejes pueden representar alto, ancho y color.
>
> Piensa en `shape` como la **etiqueta de una caja**: indica cómo está organizado el contenido, pero no qué significa ese contenido.

## Why this matters / Por qué esto importa

Before a workshop, it is useful to check the tools and the data **before** doing the mathematics. It is like checking your ingredients before you start cooking: a missing ingredient is easier to fix at the beginning than halfway through the recipe.

In this workshop we prefer **real data** whenever the real-world structure matters. Real data can contain things that perfect classroom examples often hide:

- missing values — a measurement was not recorded;
- very different numerical scales — one column may contain small decimals while another contains thousands;
- columns that do not change — later we will call these **zero-variance** columns.

A few later sections use small synthetic examples on purpose when a simple artificial example makes one mathematical idea easier to see. Those cases are explicitly identified.

> 🇪🇸 Antes de empezar con las matemáticas conviene comprobar las herramientas y los datos. Es como revisar los ingredientes antes de cocinar: es mucho más fácil resolver un problema al comienzo que a mitad de la receta.
>
> En este taller preferimos **datos reales** cuando la estructura del mundo real es importante. Los datos reales pueden tener situaciones que los ejemplos perfectos de clase suelen ocultar:
>
> - valores faltantes — una medición no fue registrada;
> - escalas numéricas muy diferentes — una columna puede contener decimales pequeños y otra valores de miles;
> - columnas que no cambian — más adelante las llamaremos columnas de **varianza cero**.
>
> Algunas secciones posteriores usan ejemplos sintéticos pequeños de forma deliberada cuando un ejemplo artificial permite ver una idea matemática con mayor claridad. Esos casos se indican explícitamente.

### Included inside the libraries / Incluidos en las librerías

These datasets are already packaged with the Python libraries, so no separate download is needed.

> 🇪🇸 Estos conjuntos de datos ya vienen incluidos con las librerías de Python, por lo que no necesitan una descarga adicional.

| Dataset / Conjunto | What it is / Qué es | Shape / Forma |
|---|---|---|
| `load_breast_cancer()` | 569 real samples, each with 30 measurements / 569 muestras reales, cada una con 30 mediciones | `(569, 30)` |
| `load_digits()` | 1,797 real handwritten-digit images / 1.797 imágenes reales de dígitos manuscritos | `(1797, 8, 8)` |
| `data.camera()`, `data.astronaut()` | Real photographs / Fotografías reales | `(512, 512)`, `(512, 512, 3)` |
| `data.immunohistochemistry()`, `data.cell()` | Real histology and microscopy images / Imágenes reales de histología y microscopía | `(512, 512, 3)`, `(660, 550)` |

### Downloaded at the start / Descargados al comienzo

These need internet access and take only a few seconds to load.

> 🇪🇸 Estos necesitan conexión a internet y tardan solo unos segundos en cargarse.

| Dataset / Conjunto | What it is / Qué es | Used for / Se usará para |
|---|---|---|
| California Housing | 20,640 real housing districts / 20.640 distritos reales de vivienda | Pseudoinverse and least squares / Pseudoinversa y mínimos cuadrados (§07) |
| NYC Taxi Trips | 6,433 real taxi trips / 6.433 viajes reales de taxi | Tensor factorization / Factorización tensorial (§10) |
| Airline Passengers | 144 months of real passenger counts / 144 meses de conteos reales de pasajeros | Recursion and forecasting / Recursión y pronóstico (§08) |

### Downloaded later / Descargados más adelante

The full video and audio are downloaded only when their sections need them. Here we only check that the video source can be reached.

> 🇪🇸 El video y el audio completos se descargan únicamente cuando sus secciones los necesitan. Aquí solo comprobamos que la fuente del video sea accesible.

| Dataset / Conjunto | What it is / Qué es | Used for / Se usará para |
|---|---|---|
| Storm video | 24 seconds, 720 frames / 24 segundos, 720 fotogramas | Video pipeline design / Diseño de pipeline de video (§05) |
| Voice recording | Five-second CC0 voice sample / Grabación de voz CC0 de cinco segundos | Audio denoising / Reducción de ruido de audio (NB11) |

### How do I know Setup worked? / ¿Cómo sé que Setup funcionó?

If you see:

`(20640, 10) (6433, 14) (144, 3)`

and a line beginning with:

`1024 bytes of video reachable`

then the main downloads and the video connection are ready.

> 🇪🇸 Si ves `(20640, 10) (6433, 14) (144, 3)` y una línea que empieza por `1024 bytes of video reachable`, las descargas principales y la conexión al video están listas.

## See it, not just its shape / Míralo, no te quedes solo con la forma

A computer can tell us that a dataset has the expected `shape`, but that does not guarantee that the data **looks reasonable**.

For example, a table may have the correct number of rows while one important column is empty. A downloaded file may also be incomplete. A quick visual check can reveal problems that a shape check cannot.

The next cell gives us three simple visual questions:

1. Do California housing locations look geographically plausible?
2. Does the taxi-fare distribution look like real measured data rather than an empty column?
3. Does airline traffic show a sensible change over time?

> 🇪🇸 Un computador puede decirnos que un conjunto tiene la `shape` esperada, pero eso no garantiza que los datos **tengan sentido**.
>
> Por ejemplo, una tabla puede tener el número correcto de filas y aun así tener una columna importante vacía. Un archivo descargado también puede estar incompleto. Una revisión visual rápida puede detectar problemas que la forma por sí sola no muestra.
>
> La siguiente celda responde tres preguntas sencillas:
>
> 1. ¿Las ubicaciones de vivienda en California parecen geográficamente razonables?
> 2. ¿La distribución de tarifas de taxi parece contener mediciones reales y no una columna vacía?
> 3. ¿El tráfico aéreo muestra un cambio razonable a lo largo del tiempo?

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

sc = axes[0].scatter(housing["longitude"], housing["latitude"],
                      c=housing["median_house_value"], cmap="viridis", s=4)
axes[0].set_title("Housing locations / Ubicaciones de vivienda")
fig.colorbar(sc, ax=axes[0], fraction=0.046)

axes[1].hist(taxis["fare"].dropna(), bins=30, color="#4C72B0")
axes[1].set_title("Taxi fares / Tarifas de taxi")
axes[1].set_xlabel("fare ($) / tarifa ($)")

by_year = flights.groupby("year")["passengers"].sum()
axes[2].plot(by_year.index, by_year.values, marker="o", color="#55A868")
axes[2].set_title("Passengers by year / Pasajeros por año")

fig.suptitle(
    "Quick visual check of real data / Revisión visual rápida de datos reales"
)
plt.tight_layout()
plt.show()

## Exercise 1 — read a shape as a sentence / Ejercicio 1 — lee una forma como una frase

A `shape` is useful only if you can explain what each number counts.

**Example / Ejemplo**

`(569, 30)` can be read as:

**EN:** “569 samples, with 30 measurements for each sample.”

**ES:** “569 muestras, con 30 mediciones para cada muestra.”

Now do the same with the datasets below.

> 🇪🇸 Una `shape` solo es útil si puedes explicar qué cuenta cada número. Usa el ejemplo anterior y describe con palabras las formas de los siguientes conjuntos.

In [ ]:
# TODO 1 / TAREA 1
# EN: Load the breast-cancer dataset and print bc.data.shape.
#     Then explain what each axis counts.
# ES: Carga el conjunto de cáncer de mama e imprime bc.data.shape.
#     Después explica qué cuenta cada eje.
#
# TODO 2 / TAREA 2
# EN: Print the shapes of load_digits().images and
#     data.immunohistochemistry().
#     Both have three axes. Do those axes mean the same thing?
# ES: Imprime las formas de load_digits().images y
#     data.immunohistochemistry().
#     Ambos tienen tres ejes. ¿Significan lo mismo esos ejes?

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

bc = load_breast_cancer()
digits_shape = load_digits().images.shape
histology_shape = data.immunohistochemistry().shape

print("Breast-cancer data / Datos de cáncer:", bc.data.shape)
print("EN: 569 samples × 30 measurements.")
print("ES: 569 muestras × 30 mediciones.")
print()

print("Digit images / Imágenes de dígitos:", digits_shape)
print("EN: 1,797 images × 8 pixels high × 8 pixels wide.")
print("ES: 1.797 imágenes × 8 píxeles de alto × 8 píxeles de ancho.")
print()

print("Histology image / Imagen histológica:", histology_shape)
print("EN: 512 pixels high × 512 pixels wide × 3 colour channels.")
print("ES: 512 píxeles de alto × 512 píxeles de ancho × 3 canales de color.")
print()

print("EN: Same number of axes does NOT mean the axes have the same meaning.")
print("ES: Tener el mismo número de ejes NO significa que los ejes tengan el mismo significado.")

<details>
<summary><strong>Why this solution works · Por qué funciona esta solución</strong></summary>

The important skill is not memorising numbers. It is learning to **translate a shape into meaning**.

- `(569, 30)` is like a spreadsheet: **569 rows of samples × 30 measurement columns**.
- `(1797, 8, 8)` is a stack of images: **image × height × width**.
- `(512, 512, 3)` is one colour image: **height × width × colour**.

So two arrays can both have three axes while describing completely different things.

**EN:** Shape tells us how data is arranged; the dataset tells us what the axes mean.

> 🇪🇸 La habilidad importante no es memorizar números, sino **traducir una forma a significado**.
>
> - `(569, 30)` se parece a una hoja de cálculo: **569 filas de muestras × 30 columnas de mediciones**.
> - `(1797, 8, 8)` es una pila de imágenes: **imagen × alto × ancho**.
> - `(512, 512, 3)` es una imagen a color: **alto × ancho × color**.
>
> Dos arreglos pueden tener tres ejes y representar cosas completamente distintas.
>
> **ES:** La forma indica cómo están organizados los datos; el conjunto de datos nos dice qué significan los ejes.

</details>

## Exercise 2 — find two real-data clues / Ejercicio 2 — encuentra dos pistas en datos reales

Real datasets are rarely perfectly clean. In this exercise you will find:

1. a **missing-data clue** in the housing table, and
2. a **time pattern** in the taxi table.

You do not need advanced statistics. We are simply asking: “Is anything missing?” and “At what hour do we see the most trips?”

> 🇪🇸 Los conjuntos de datos reales rara vez están perfectamente limpios. En este ejercicio encontrarás:
>
> 1. una **pista de datos faltantes** en la tabla de vivienda, y
> 2. un **patrón temporal** en la tabla de taxis.
>
> No necesitas estadística avanzada. Solo preguntaremos: “¿Falta algún dato?” y “¿A qué hora observamos más viajes?”

In [ ]:
# TODO 3 / TAREA 3
# EN: Print housing.shape and count the missing values in each column.
#     Which column has missing values? How many?
# ES: Imprime housing.shape y cuenta los valores faltantes de cada columna.
#     ¿Qué columna tiene valores faltantes? ¿Cuántos?
#
# TODO 4 / TAREA 4
# EN: Convert taxis["pickup"] to datetime, extract the hour,
#     and find the hour with the most trips.
# ES: Convierte taxis["pickup"] a fecha y hora, extrae la hora
#     y encuentra la hora con mayor cantidad de viajes.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("Housing shape / Forma de vivienda:", housing.shape)

missing = housing.isnull().sum()
missing = missing[missing > 0]
print("\nMissing values / Valores faltantes:")
print(missing)

hour = pd.to_datetime(taxis["pickup"]).dt.hour
busiest_hour = int(hour.value_counts().idxmax())

print("\nBusiest pickup hour / Hora con más recogidas:", busiest_hour)
print("EN: The busiest hour is 18:00 (6:00 p.m.) in this dataset.")
print("ES: La hora con más recogidas es 18:00 (6:00 p. m.) en este conjunto.")

print("\nEN: Missing values are not an error in Python; they are a property of the real data that we must handle.")
print("ES: Los valores faltantes no son un error de Python; son una característica de los datos reales que debemos tratar.")

<details>
<summary><strong>Why this solution works · Por qué funciona esta solución</strong></summary>

`housing.shape == (20640, 10)` means **20,640 rows and 10 columns**. Think of it as an Excel table.

The next check finds **207 missing values in `total_bedrooms`**. A missing value means that, for those rows, that particular measurement is unavailable. Later, before matrix calculations, we must decide how to handle those rows.

For taxis, a timestamp such as a pickup date contains several pieces of information. Extracting the **hour** lets us group trips by time of day. In this dataset, hour `18` has the most pickups.

> 🇪🇸 `housing.shape == (20640, 10)` significa **20.640 filas y 10 columnas**. Puedes imaginarlo como una tabla de Excel.
>
> La siguiente comprobación encuentra **207 valores faltantes en `total_bedrooms`**. Un valor faltante significa que, para esas filas, esa medición concreta no está disponible. Más adelante, antes de hacer cálculos matriciales, tendremos que decidir cómo tratar esas filas.
>
> En los taxis, una fecha y hora contiene varias piezas de información. Extraer la **hora** nos permite agrupar los viajes según el momento del día. En este conjunto, la hora `18` tiene la mayor cantidad de recogidas.

</details>

## A note on how these notebooks work / Cómo funcionan estos cuadernos

Every notebook is **self-contained**. That means each notebook brings the tools and data it needs, so you can open Notebook 07 directly without first running Notebooks 00–06.

Because of that design, you may see the same download address in more than one notebook. That repetition is intentional: each notebook should work on its own.

The repository stores notebooks with **no saved outputs**. When you press **Run all / Ejecutar todo**, the numbers, plots, and widgets you see are generated by your own Colab session.

If your result differs from an expected result, treat that difference as a clue to investigate — not something to ignore.

> 🇪🇸 Cada cuaderno es **autónomo**. Esto significa que cada notebook carga las herramientas y los datos que necesita, por lo que puedes abrir directamente el Notebook 07 sin ejecutar antes los Notebooks 00–06.
>
> Por ese diseño, algunas direcciones de descarga aparecen en más de un cuaderno. La repetición es intencional: cada notebook debe funcionar por sí solo.
>
> El repositorio guarda los cuadernos **sin resultados ejecutados**. Cuando presionas **Run all / Ejecutar todo**, los números, gráficos y widgets que aparecen son generados por tu propia sesión de Colab.
>
> Si tu resultado es diferente del esperado, toma esa diferencia como una pista para investigar, no como algo que debas ignorar.

## What just happened / Qué acaba de pasar

You are now ready to start the workshop. More importantly, you already used four habits that will appear again and again:

- **Check the environment:** Colab can reach the data and the video tools are available.
- **Read shapes as meaning:** `(20640, 10)` is not just two numbers; it means rows and columns.
- **Look at the data:** a graph can reveal problems that a shape check misses.
- **Expect imperfect real data:** missing values are something to understand and handle.

**One-sentence takeaway:** before asking a model to learn from data, first make sure you understand what the data contains and how it is organised.

> 🇪🇸 Ya estás listo para comenzar el taller. Más importante aún, ya utilizaste cuatro hábitos que aparecerán una y otra vez:
>
> - **Comprobar el entorno:** Colab puede acceder a los datos y las herramientas de video están disponibles.
> - **Leer las formas con significado:** `(20640, 10)` no son solo dos números; representan filas y columnas.
> - **Mirar los datos:** un gráfico puede mostrar problemas que una revisión de `shape` no detecta.
> - **Esperar imperfecciones en datos reales:** los valores faltantes son algo que debemos entender y tratar.
>
> **Idea en una frase:** antes de pedirle a un modelo que aprenda de los datos, asegúrate primero de entender qué contienen y cómo están organizados.

---

## Done with this section

> 🇪🇸 **Fin de esta sección.**

Next up: **01 · What a tensor is** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/01-what-a-tensor-is.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)